# Gap-Level Policy Compliance Detection using SetFit

**SetFit** (Sentence Transformer Fine-tuning) — few-shot contrastive learning approach.

**Why SetFit over fine-tuning XLM-R?**
- Designed for few-shot: works well with <1000 samples (our dataset: ~891)
- Contrastive learning generates richer embeddings from limited data
- Built-in multi-label support (`one-vs-rest`, `multi-output`, `classifier-chain`)
- Multilingual via sentence transformer body
- Order of magnitude faster training
- No prompts or verbalizers needed

**Model**: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` → 16 multi-label outputs
**Standards**: NCA ECC-2:2024 + ISO 27001:2022

| Gap ID | Domain | Description |
|--------|--------|-------------|
| GAP_PP_001-008 | Password Policy (ECC 2-2) | Complexity, expiration, lockout, MFA, PAM, encryption, review, roles |
| GAP_RA_001-008 | Risk Assessment (ECC 1-5) | Methodology, identification, scales, treatment, triggers, register, review, integration |

In [ ]:
# Install required packages
%pip install setfit sentence-transformers datasets scikit-learn torch -q

In [ ]:
import json
import random
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Load Dataset & Define Labels

In [ ]:
# ========== GAP LABELS (16 gaps) ==========
GAP_LABELS = [
    "GAP_PP_001", "GAP_PP_002", "GAP_PP_003", "GAP_PP_004",
    "GAP_PP_005", "GAP_PP_006", "GAP_PP_007", "GAP_PP_008",
    "GAP_RA_001", "GAP_RA_002", "GAP_RA_003", "GAP_RA_004",
    "GAP_RA_005", "GAP_RA_006", "GAP_RA_007", "GAP_RA_008",
]
NUM_GAPS = len(GAP_LABELS)

# ========== LOAD DATASET ==========
dataset_path = "datasets/processed/incremental_samples.jsonl"
samples = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

print(f"Loaded {len(samples)} samples from JSONL")

# Extract texts and multi-hot labels
texts = [s["policy_excerpt"] for s in samples]
labels = np.array([
    [s["gap_labels"].get(g, 0) for g in GAP_LABELS] for s in samples
], dtype=np.float32)

print(f"\nGap frequencies (out of {len(samples)} samples):")
for i, g in enumerate(GAP_LABELS):
    count = int(labels[:, i].sum())
    bar = "\u2588" * int(count / len(samples) * 30)
    print(f"  {g}: {count:>3} ({count/len(samples):>5.1%}) {bar}")

print(f"\nAvg gaps per sample: {labels.sum(axis=1).mean():.1f}")
print(f"Label density: {labels.mean():.2%}")

## 2. Train / Val / Test Split

In [ ]:
# Stratify on gap count (binned) to keep distribution balanced
gap_counts = labels.sum(axis=1).astype(int)
strat_bins = np.where(gap_counts == 0, 0,
             np.where(gap_counts <= 2, 1,
             np.where(gap_counts <= 5, 2, 3)))

# Split: 70% train, 15% val, 15% test
indices = list(range(len(samples)))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, random_state=SEED, stratify=strat_bins
)
temp_strat = strat_bins[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, random_state=SEED, stratify=temp_strat
)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
print(f"Train gap distribution: {np.bincount(strat_bins[train_idx], minlength=4)}")
print(f"Val gap distribution:   {np.bincount(strat_bins[val_idx], minlength=4)}")
print(f"Test gap distribution:  {np.bincount(strat_bins[test_idx], minlength=4)}")

## 3. Prepare SetFit Datasets

SetFit expects a HuggingFace `Dataset` with `text` and `label` columns.
For multi-label, `label` is a list of ints (multi-hot vector).

In [ ]:
from datasets import Dataset

def make_hf_dataset(idxs):
    """Convert our indices into a HuggingFace Dataset for SetFit."""
    return Dataset.from_dict({
        "text": [texts[i] for i in idxs],
        "label": [labels[i].astype(int).tolist() for i in idxs],
    })

train_dataset = make_hf_dataset(train_idx)
val_dataset = make_hf_dataset(val_idx)
test_dataset = make_hf_dataset(test_idx)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset:   {len(val_dataset)} samples")
print(f"Test dataset:  {len(test_dataset)} samples")
print(f"\nSample entry:")
print(f"  text: {train_dataset[0]['text'][:120]}...")
print(f"  label: {train_dataset[0]['label']}")

## 4. Initialize SetFit Model

Using `paraphrase-multilingual-mpnet-base-v2` as the sentence transformer body:
- 278M parameters, supports 50+ languages (including Arabic & English)
- Produces 768-dim embeddings
- Proven performance with SetFit in multilingual settings

**Multi-label strategy**: `one-vs-rest` — trains one LogisticRegression per label.

In [ ]:
from setfit import SetFitModel, Trainer, TrainingArguments

# Multilingual sentence transformer — supports English + Arabic
ST_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

model = SetFitModel.from_pretrained(
    ST_MODEL,
    multi_target_strategy="one-vs-rest",
)

print(f"Model: {ST_MODEL}")
print(f"Multi-target strategy: one-vs-rest")
print(f"Number of labels: {NUM_GAPS}")

## 5. Training

SetFit training has two phases:
1. **Contrastive fine-tuning** of the sentence transformer body on text pairs
2. **Classification head training** on the resulting embeddings

Key parameters:
- `num_epochs`: Epochs for contrastive learning (body fine-tuning)
- `num_iterations`: Number of text pairs generated per class for contrastive learning
- `batch_size`: Batch size for contrastive training

In [ ]:
N = len(train_dataset)

# Adaptive config based on dataset size
if N <= 300:
    NUM_ITERATIONS = 30
    NUM_EPOCHS = 3
    BATCH_SIZE = 16
elif N <= 700:
    NUM_ITERATIONS = 25
    NUM_EPOCHS = 2
    BATCH_SIZE = 32
else:
    NUM_ITERATIONS = 20
    NUM_EPOCHS = 2
    BATCH_SIZE = 32

print(f"Training config (N={N} train samples):")
print(f"  NUM_ITERATIONS = {NUM_ITERATIONS} (text pairs per class)")
print(f"  NUM_EPOCHS = {NUM_EPOCHS} (contrastive learning epochs)")
print(f"  BATCH_SIZE = {BATCH_SIZE}")

args = TrainingArguments(
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    num_iterations=NUM_ITERATIONS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    column_mapping={"text": "text", "label": "label"},
)

In [ ]:
# Train!
print("=" * 65)
print("TRAINING: SetFit Multi-Label Gap Detection")
print("=" * 65)

trainer.train()

print("\nTraining complete!")

## 6. Evaluation

In [ ]:
# ========== VALIDATION SET ==========
val_metrics = trainer.evaluate(val_dataset)
print(f"Validation metrics: {val_metrics}")

In [ ]:
# ========== TEST SET — DETAILED EVALUATION ==========
test_preds = model.predict(test_dataset["text"])
test_preds = np.array(test_preds)
test_labels = np.array(test_dataset["label"])

# Metrics
hamming_acc = (test_preds == test_labels).mean()
exact_match = (test_preds == test_labels).all(axis=1).mean()
macro_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)
micro_f1 = f1_score(test_labels, test_preds, average='micro', zero_division=0)
weighted_f1 = f1_score(test_labels, test_preds, average='weighted', zero_division=0)

print("=" * 65)
print("TEST SET RESULTS")
print("=" * 65)
print(f"Hamming Accuracy: {hamming_acc:.4f}")
print(f"Exact Match:      {exact_match:.4f}")
print(f"Macro F1:         {macro_f1:.4f}")
print(f"Micro F1:         {micro_f1:.4f}")
print(f"Weighted F1:      {weighted_f1:.4f}")

print(f"\n{'=' * 65}")
print("PER-LABEL CLASSIFICATION REPORT")
print("=" * 65)
print(classification_report(
    test_labels, test_preds,
    target_names=GAP_LABELS,
    zero_division=0
))

## 7. Compare with XLM-R Baseline

| Metric | XLM-R (current) | SetFit |
|--------|:---:|:---:|
| Test samples | ~134 | ~134 |
| Test Macro F1 | 0.5962 | **?** |
| Training time | ~20 min | ~2 min |

## 8. Alternative Strategies

If `one-vs-rest` doesn't give best results, try:

```python
# Capture label correlations (PP gaps often co-occur)
model = SetFitModel.from_pretrained(ST_MODEL, multi_target_strategy="classifier-chain")

# Or use a differentiable PyTorch head instead of sklearn
model = SetFitModel.from_pretrained(
    ST_MODEL,
    multi_target_strategy="one-vs-rest",
    use_differentiable_head=True,
    head_params={"out_features": 16},
)
```

## 9. Save Model

In [ ]:
import os

save_dir = "models/gap_detector_setfit"
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)

# Save metadata
metadata = {
    "model_type": "setfit",
    "st_model": ST_MODEL,
    "multi_target_strategy": "one-vs-rest",
    "gap_labels": GAP_LABELS,
    "num_gaps": NUM_GAPS,
    "dataset_size": len(samples),
    "train_size": len(train_idx),
    "test_macro_f1": float(macro_f1),
    "test_micro_f1": float(micro_f1),
    "test_exact_match": float(exact_match),
    "training_args": {
        "num_iterations": NUM_ITERATIONS,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
    },
}

with open(f"{save_dir}/config_meta.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Model saved to: {save_dir}/")
print(f"Test Macro F1: {macro_f1:.4f}")

## 10. Inference Example

In [ ]:
# Load saved model
loaded_model = SetFitModel.from_pretrained(save_dir)

# Test inference with sample texts
test_texts = [
    "Passwords must be at least 8 characters. No MFA is required.",
    "Risk assessments are conducted annually using a documented methodology with likelihood and impact scales.",
    "The policy has no password requirements and no risk management framework.",
]

preds = loaded_model.predict(test_texts)
preds = np.array(preds)

for i, text in enumerate(test_texts):
    gaps = [g for j, g in enumerate(GAP_LABELS) if preds[i][j] == 1]
    print(f"\nText: {text[:80]}...")
    print(f"  Gaps detected ({len(gaps)}): {gaps if gaps else 'None (compliant)'}")